# Pseudo-Top Gated Regime Model

This notebook applies the pseudo-top lesson from the prior screen: pseudo-top estimates are useful latent state, but should control shrinkage/gating rather than enter only as raw tree columns.

## 1. Purpose And Leakage Rules

Formation columns are local train-only labels. This notebook trains top-elevation proxies inside each validation fold, derives pseudo-top uncertainty features on held-out rows, fits safe/typewell mean models, and applies residual corrections with conservative gates.

Legal candidates never use direct formation columns as predictors. Oracle diagnostics are excluded from this pass so the comparison stays focused on package-relevant behavior.

## 2. Setup

In [1]:
"Configure imports, paths, and deterministic run constants."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

ROOT = Path.cwd()
if not (ROOT / "context").exists():
    ROOT = next(parent for parent in ROOT.parents if (parent / "context").exists())

RUN_DATE = "2026-06-30"
RUN_TAG = "pseudo_top_gated_regime"
RUN_ID = f"{RUN_DATE}__{RUN_TAG}"
SEED = 20260630
FORMATION_COLS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PRIMARY_FOLD = "strict_hybrid_fold"
STRESS_FOLD = "analog_distance_fold"
MAX_TARGET_ROWS_PER_WELL = 120
ENSEMBLE_SEEDS = [20260630, 20260631]

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
TABLE_DIR = ROOT / "artifacts" / "tables"
REPORT_DIR = ROOT / "artifacts" / "reports"
RUN_DIR = ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for directory in [TABLE_DIR, REPORT_DIR, RUN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_id": RUN_ID,
    "seed": SEED,
    "formation_cols": FORMATION_COLS,
    "primary_fold": PRIMARY_FOLD,
    "stress_fold": STRESS_FOLD,
    "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL,
    "ensemble_seeds": ENSEMBLE_SEEDS,
    "active_public_best_rmse": 14.472,
    "safe_typewell_reference_rmse": 13.4506,
    "safe_typewell_reference_stress_rmse": 13.8870,
}
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as handle:
    json.dump(CONFIG, handle, indent=2)

print(ROOT)
print(RUN_DIR)

C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\logs\runs\2026-06-30__pseudo_top_gated_regime


## 3. Fold-Safe Pseudo-Top Frame

In [2]:
"Build target-zone rows, legal predictors, and fold-safe pseudo-top uncertainty features."

folds = pd.read_csv(TABLE_DIR / "validation__strict_cv_fold_assignments__2026-06-26.csv")
descriptors = pd.read_csv(TABLE_DIR / "model__typewell_alignment_feature_descriptors__2026-06-23.csv")
well_meta = folds.merge(descriptors, on="wellname", how="left").drop_duplicates("wellname").reset_index(drop=True)

DESCRIPTOR_COLS = [
    "known_rows", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300",
    "xy_span_known", "z_drop_known", "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr",
    "typewell_best_lag_tvt", "typewell_best_lag_mae", "selected_analog_distance", "extrapolation_stress_score",
    "analog_stress_score", "combined_stress_score", "descriptor_cluster",
]


def safe_divide(num, den):
    return num / (den + 1e-9)


def load_typewell(directory, wellname):
    path = Path(directory) / f"{wellname}__typewell.csv"
    if not path.exists():
        return pd.DataFrame(columns=["TVT", "GR", "Geology"])
    return pd.read_csv(path)


def interp_typewell_gr(typewell, tvt_values):
    if typewell.empty or not {"TVT", "GR"}.issubset(typewell.columns):
        return np.full(len(tvt_values), np.nan)
    clean = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(clean) < 5:
        return np.full(len(tvt_values), np.nan)
    return np.interp(np.asarray(tvt_values, dtype=float), clean["TVT"].to_numpy(), clean["GR"].to_numpy(), left=np.nan, right=np.nan)


def build_target_rows(csv_path, meta_row):
    df = pd.read_csv(csv_path).sort_values("MD").reset_index(drop=True)
    known_mask = df["TVT_input"].notna()
    if known_mask.sum() == 0 or (~known_mask).sum() == 0:
        return pd.DataFrame()
    last_known_idx = int(np.flatnonzero(known_mask.to_numpy())[-1])
    known = df.loc[:last_known_idx].copy()
    target = df.loc[last_known_idx + 1:].copy()
    target = target.loc[target["TVT"].notna()].copy()
    if target.empty:
        return pd.DataFrame()
    if len(target) > MAX_TARGET_ROWS_PER_WELL:
        sample_idx = np.linspace(0, len(target) - 1, MAX_TARGET_ROWS_PER_WELL).round().astype(int)
        target = target.iloc[np.unique(sample_idx)].copy()

    typewell = load_typewell(csv_path.parent, meta_row["wellname"])
    last = known.iloc[-1]
    first = df.iloc[0]
    known_gr = known["GR"].astype(float).clip(0, 300)
    known_gr_finite = known_gr.dropna()
    gr_fill = float(known_gr.tail(300).median()) if known_gr.notna().any() else 0.0
    target_gr = target["GR"].astype(float).clip(0, 300).fillna(gr_fill)
    known_md_span = float(known["MD"].iloc[-1] - known["MD"].iloc[0]) if len(known) > 1 else 0.0
    known_tvt_delta = float(known["TVT_input"].iloc[-1] - known["TVT_input"].iloc[0]) if len(known) > 1 else 0.0
    known_xy_span = float(np.hypot(known["X"].iloc[-1] - known["X"].iloc[0], known["Y"].iloc[-1] - known["Y"].iloc[0])) if len(known) > 1 else 0.0
    known_z_drop = float(known["Z"].iloc[-1] - known["Z"].iloc[0]) if len(known) > 1 else 0.0

    target = target.reset_index(drop=False).rename(columns={"index": "row_idx_original"})
    out = pd.DataFrame({
        "wellname": meta_row["wellname"],
        "row_idx_original": target["row_idx_original"].astype(float),
        "MD": target["MD"].astype(float),
        "X": target["X"].astype(float),
        "Y": target["Y"].astype(float),
        "Z": target["Z"].astype(float),
        "GR": target_gr.to_numpy(),
        "TVT": target["TVT"].astype(float),
        "last_known_tvt_feature": float(last["TVT"]),
        "last_known_md": float(last["MD"]),
        "last_known_x": float(last["X"]),
        "last_known_y": float(last["Y"]),
        "last_known_z": float(last["Z"]),
        "first_md": float(first["MD"]),
        "state_known_tvt_last": float(last["TVT_input"] if pd.notna(last.get("TVT_input", np.nan)) else last["TVT"]),
        "state_known_tvt_slope": known_tvt_delta / max(abs(known_md_span), 1.0),
        "state_known_gr_coverage": float(known_gr.notna().mean()) if len(known_gr) else 0.0,
        "state_known_gr_last": float(known_gr_finite.iloc[-1]) if len(known_gr_finite) else gr_fill,
        "state_known_gr_iqr": float(known_gr_finite.quantile(0.75) - known_gr_finite.quantile(0.25)) if len(known_gr_finite) >= 4 else 0.0,
        "state_known_xy_span": known_xy_span,
        "state_known_z_drop": known_z_drop,
        "state_known_md_span": known_md_span,
    })
    out["target_delta"] = out["TVT"] - out["last_known_tvt_feature"]
    out["md_from_last_known"] = out["MD"] - out["last_known_md"]
    out["md_from_start"] = out["MD"] - out["first_md"]
    out["eval_progress"] = safe_divide(np.arange(1, len(out) + 1), len(out))
    out["eval_progress_sq"] = out["eval_progress"] ** 2
    out["dx_from_last"] = out["X"] - out["last_known_x"]
    out["dy_from_last"] = out["Y"] - out["last_known_y"]
    out["dz_from_last"] = out["Z"] - out["last_known_z"]
    out["horizontal_from_last"] = np.sqrt(out["dx_from_last"] ** 2 + out["dy_from_last"] ** 2)
    out["distance_from_last"] = np.sqrt(out["horizontal_from_last"] ** 2 + out["dz_from_last"] ** 2)
    out["distance_log1p"] = np.log1p(out["distance_from_last"].clip(lower=0))
    out["tvt_linear_proxy"] = out["last_known_tvt_feature"] + out["md_from_last_known"]
    best_lag = float(meta_row.get("typewell_best_lag_tvt", 0.0) if pd.notna(meta_row.get("typewell_best_lag_tvt", np.nan)) else 0.0)
    out["typewell_gr_linear_tvt"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy())
    out["typewell_gr_lag_best"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy() + best_lag)
    out["gr_minus_typewell_linear"] = out["GR"] - out["typewell_gr_linear_tvt"]
    out["gr_minus_typewell_bestlag"] = out["GR"] - out["typewell_gr_lag_best"]

    for formation in FORMATION_COLS:
        out[f"formation_top_{formation}"] = target[formation].astype(float).to_numpy()
        out[f"label_signed_offset_{formation}"] = out["Z"] - out[f"formation_top_{formation}"]
        out[f"label_abs_offset_{formation}"] = np.abs(out[f"label_signed_offset_{formation}"])
    abs_label_cols = [f"label_abs_offset_{formation}" for formation in FORMATION_COLS]
    out["label_nearest_abs_offset"] = out[abs_label_cols].min(axis=1)
    out["label_nearest_formation"] = out[abs_label_cols].idxmin(axis=1).str.replace("label_abs_offset_", "", regex=False)

    for col in [PRIMARY_FOLD, STRESS_FOLD, "descriptor_cluster_fold", "extrapolation_fold"]:
        out[col] = int(meta_row[col])
    for col in DESCRIPTOR_COLS:
        out[col] = float(meta_row[col]) if pd.notna(meta_row[col]) else np.nan
    return out


frames = []
for meta_row in well_meta.to_dict(orient="records"):
    csv_path = TRAIN_DIR / f"{meta_row['wellname']}__horizontal_well.csv"
    if csv_path.exists():
        frame = build_target_rows(csv_path, meta_row)
        if not frame.empty:
            frames.append(frame)
data = pd.concat(frames, ignore_index=True)

SAFE_ROW_COLS = [
    "row_idx_original", "MD", "X", "Y", "Z", "GR", "last_known_tvt_feature", "last_known_md",
    "md_from_last_known", "md_from_start", "eval_progress", "eval_progress_sq", "dx_from_last", "dy_from_last",
    "dz_from_last", "horizontal_from_last", "distance_from_last", "distance_log1p",
]
SAFE_FEATURE_COLS = [col for col in [*SAFE_ROW_COLS, *DESCRIPTOR_COLS] if col in data.columns]
TYPEWELL_FEATURE_COLS = ["tvt_linear_proxy", "typewell_gr_linear_tvt", "typewell_gr_lag_best", "gr_minus_typewell_linear", "gr_minus_typewell_bestlag"]
TYPEWELL_FEATURE_COLS = [col for col in TYPEWELL_FEATURE_COLS if col in data.columns]
STATE_FEATURE_COLS = [
    "state_known_tvt_last", "state_known_tvt_slope", "state_known_gr_coverage", "state_known_gr_last",
    "state_known_gr_iqr", "state_known_xy_span", "state_known_z_drop", "state_known_md_span",
]

proxy_predictor_cols = list(dict.fromkeys([*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS, *STATE_FEATURE_COLS]))
proxy_params = {
    "objective": "reg:squarederror",
    "n_estimators": 220,
    "learning_rate": 0.045,
    "max_depth": 3,
    "min_child_weight": 90,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.20,
    "reg_lambda": 10.0,
    "tree_method": "hist",
    "n_jobs": 4,
    "verbosity": 0,
}


def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def softmax_neg_abs(abs_matrix, scale=50.0):
    logits = -abs_matrix / scale
    logits = logits - np.nanmax(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.clip(exp_logits.sum(axis=1, keepdims=True), 1e-12, None)


def add_pseudo_top_features(frame, fold_col):
    out = frame.copy()
    quality_rows = []
    X_all = out[proxy_predictor_cols].replace([np.inf, -np.inf], np.nan)
    for formation in FORMATION_COLS:
        out[f"pseudo_top_elev_{formation}__{fold_col}"] = np.nan
        out[f"pseudo_top_std_{formation}__{fold_col}"] = np.nan
    for fold in sorted(out[fold_col].dropna().unique()):
        valid_mask = out[fold_col].eq(fold)
        train_mask = ~valid_mask
        X_valid = X_all.loc[valid_mask]
        for formation in FORMATION_COLS:
            target_col = f"formation_top_{formation}"
            label_train_mask = train_mask & out[target_col].notna()
            label_valid_mask = valid_mask & out[target_col].notna()
            seed_predictions = []
            if label_train_mask.sum() >= 20:
                X_train = X_all.loc[label_train_mask]
                y_train = out.loc[label_train_mask, target_col].astype(float)
                for seed in ENSEMBLE_SEEDS:
                    model = XGBRegressor(**proxy_params, random_state=seed)
                    model.fit(X_train, y_train, verbose=False)
                    seed_predictions.append(model.predict(X_valid))
                pred_matrix = np.column_stack(seed_predictions)
                pred_mean = pred_matrix.mean(axis=1)
                pred_std = pred_matrix.std(axis=1)
            else:
                pred_mean = np.full(valid_mask.sum(), float(out[target_col].median()))
                pred_std = np.zeros(valid_mask.sum())
            out.loc[valid_mask, f"pseudo_top_elev_{formation}__{fold_col}"] = pred_mean
            out.loc[valid_mask, f"pseudo_top_std_{formation}__{fold_col}"] = pred_std
            if label_valid_mask.sum() > 0:
                pred_series = pd.Series(pred_mean, index=out.loc[valid_mask].index).loc[label_valid_mask]
                y_valid = out.loc[label_valid_mask, target_col].astype(float)
                quality_rows.append({
                    "fold_column": fold_col,
                    "fold": int(fold),
                    "formation": formation,
                    "top_rmse": rmse(y_valid, pred_series),
                    "top_mae": float(mean_absolute_error(y_valid, pred_series)),
                    "mean_ensemble_std": float(np.mean(pred_std)),
                    "n_train": int(label_train_mask.sum()),
                    "n_valid": int(label_valid_mask.sum()),
                })
    for formation in FORMATION_COLS:
        top_col = f"pseudo_top_elev_{formation}__{fold_col}"
        std_col = f"pseudo_top_std_{formation}__{fold_col}"
        out[top_col] = out[top_col].fillna(float(out[f"formation_top_{formation}"].median()))
        out[std_col] = out[std_col].fillna(float(out[std_col].median()) if out[std_col].notna().any() else 0.0)
        out[f"pseudo_signed_offset_{formation}__{fold_col}"] = out["Z"] - out[top_col]
        out[f"pseudo_abs_offset_{formation}__{fold_col}"] = np.abs(out[f"pseudo_signed_offset_{formation}__{fold_col}"])
    abs_cols = [f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS]
    std_cols = [f"pseudo_top_std_{formation}__{fold_col}" for formation in FORMATION_COLS]
    abs_matrix = out[abs_cols].to_numpy(float)
    probs = softmax_neg_abs(abs_matrix)
    sorted_abs = np.sort(abs_matrix, axis=1)
    sorted_probs = np.sort(probs, axis=1)[:, ::-1]
    nearest_idx = np.argmin(abs_matrix, axis=1)
    out[f"pseudo_nearest_abs_offset__{fold_col}"] = sorted_abs[:, 0]
    out[f"pseudo_second_abs_offset__{fold_col}"] = sorted_abs[:, 1]
    out[f"pseudo_top_margin__{fold_col}"] = sorted_abs[:, 1] - sorted_abs[:, 0]
    out[f"pseudo_top_confidence__{fold_col}"] = 1.0 / (1.0 + np.exp(-out[f"pseudo_top_margin__{fold_col}"].clip(-100, 100) / 25.0))
    out[f"pseudo_mean_abs_offset__{fold_col}"] = abs_matrix.mean(axis=1)
    out[f"pseudo_min_std__{fold_col}"] = out[std_cols].min(axis=1)
    out[f"pseudo_mean_std__{fold_col}"] = out[std_cols].mean(axis=1)
    out[f"pseudo_top1_prob__{fold_col}"] = sorted_probs[:, 0]
    out[f"pseudo_top2_prob__{fold_col}"] = sorted_probs[:, 1]
    out[f"pseudo_top_entropy__{fold_col}"] = -(probs * np.log(probs + 1e-12)).sum(axis=1)
    out[f"pseudo_nearest_formation__{fold_col}"] = [FORMATION_COLS[int(idx)] for idx in nearest_idx]
    out[f"pseudo_nearest_matches_label__{fold_col}"] = (out[f"pseudo_nearest_formation__{fold_col}"] == out["label_nearest_formation"]).astype(float)
    return out, pd.DataFrame(quality_rows)


pseudo_quality_parts = []
for active_fold in [PRIMARY_FOLD, STRESS_FOLD]:
    data, fold_quality = add_pseudo_top_features(data, active_fold)
    pseudo_quality_parts.append(fold_quality)
pseudo_quality = pd.concat(pseudo_quality_parts, ignore_index=True)

feature_preview_path = TABLE_DIR / "model__pseudo_top_gated_regime_feature_preview__2026-06-30.csv"
data.head(500).to_csv(feature_preview_path, index=False)
shutil.copy2(feature_preview_path, RUN_DIR / feature_preview_path.name)

print(data.shape)
print(pseudo_quality.groupby(["fold_column", "formation"]).agg(top_rmse=("top_rmse", "mean")).reset_index().head(12).to_string(index=False))

(92760, 152)
         fold_column formation  top_rmse
analog_distance_fold      ANCC 36.143833
analog_distance_fold     ASTNL 36.206339
analog_distance_fold     ASTNU 37.919348
analog_distance_fold      BUDA 34.533433
analog_distance_fold     EGFDL 34.615113
analog_distance_fold     EGFDU 34.296069
  strict_hybrid_fold      ANCC 35.299203
  strict_hybrid_fold     ASTNL 35.324031
  strict_hybrid_fold     ASTNU 36.589313
  strict_hybrid_fold      BUDA 33.635108
  strict_hybrid_fold     EGFDL 33.407593
  strict_hybrid_fold     EGFDU 33.613043


## 4. Mean And Residual Experts

In [3]:
"Define mean models, residual experts, and pseudo-top gate features."

BASE_FEATURE_COLS = list(dict.fromkeys([*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS]))
STATE_GATE_COLS = [
    "state_known_tvt_slope", "state_known_gr_coverage", "state_known_gr_last", "state_known_gr_iqr",
    "state_known_xy_span", "state_known_z_drop", "state_known_md_span",
    "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr", "selected_analog_distance",
    "extrapolation_stress_score", "analog_stress_score", "combined_stress_score",
]
STATE_GATE_COLS = [col for col in STATE_GATE_COLS if col in data.columns]


def pseudo_cols_for_fold(fold_col):
    residual_cols = [
        *[f"pseudo_signed_offset_{formation}__{fold_col}" for formation in FORMATION_COLS],
        *[f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS],
        f"pseudo_nearest_abs_offset__{fold_col}",
        f"pseudo_second_abs_offset__{fold_col}",
        f"pseudo_mean_abs_offset__{fold_col}",
    ]
    uncertainty_cols = [
        *[f"pseudo_top_std_{formation}__{fold_col}" for formation in FORMATION_COLS],
        f"pseudo_top_margin__{fold_col}",
        f"pseudo_top_confidence__{fold_col}",
        f"pseudo_min_std__{fold_col}",
        f"pseudo_mean_std__{fold_col}",
        f"pseudo_top1_prob__{fold_col}",
        f"pseudo_top2_prob__{fold_col}",
        f"pseudo_top_entropy__{fold_col}",
    ]
    return residual_cols, uncertainty_cols


def metric_row(actual, pred, baseline):
    return {
        "rmse": rmse(actual, pred),
        "mae": float(mean_absolute_error(actual, pred)),
        "r2": float(r2_score(actual, pred)),
        "last_known_hold_rmse": rmse(actual, baseline),
    }


def make_xgb_mean(kind):
    if kind == "xgb_pseudohuber_mean":
        return XGBRegressor(
            objective="reg:pseudohubererror", n_estimators=650, learning_rate=0.032, max_depth=4,
            min_child_weight=100, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.15,
            reg_lambda=7.0, gamma=0.05, tree_method="hist", random_state=SEED, n_jobs=4, verbosity=0,
        )
    if kind == "xgb_square_mean":
        return XGBRegressor(
            objective="reg:squarederror", n_estimators=520, learning_rate=0.036, max_depth=5,
            min_child_weight=90, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.20,
            reg_lambda=8.0, gamma=0.10, tree_method="hist", random_state=SEED, n_jobs=4, verbosity=0,
        )
    raise ValueError(kind)


def make_linear_residual(kind):
    if kind == "ridge_pseudo_state_residual":
        return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=120.0))
    if kind == "huber_pseudo_state_residual":
        return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), HuberRegressor(alpha=0.0001, epsilon=1.35, max_iter=900))
    raise ValueError(kind)


def fit_mean_predict(mean_name, X_train, y_train_delta, X_valid):
    if mean_name == "huber_mean":
        model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), HuberRegressor(alpha=0.0001, epsilon=1.35, max_iter=900))
        model.fit(X_train, y_train_delta)
        return model.predict(X_valid), model
    model = make_xgb_mean(mean_name)
    model.fit(X_train, y_train_delta, verbose=False)
    return model.predict(X_valid), model


def make_gate_weights(train_gate, valid_gate, schedule, max_weight):
    cols = list(train_gate.columns)
    train_medians = train_gate.median(numeric_only=True)
    train_filled = train_gate.fillna(train_medians)
    valid_filled = valid_gate.fillna(train_medians)
    margin_col = [col for col in cols if "pseudo_top_margin" in col][0]
    conf_col = [col for col in cols if "pseudo_top_confidence" in col][0]
    entropy_col = [col for col in cols if "pseudo_top_entropy" in col][0]
    std_col = [col for col in cols if "pseudo_mean_std" in col][0]
    nearest_col = [col for col in cols if "pseudo_nearest_abs_offset" in col][0]
    margin_ref = float(train_filled[margin_col].quantile(0.75))
    entropy_ref = float(train_filled[entropy_col].quantile(0.50))
    std_ref = float(train_filled[std_col].quantile(0.75))
    nearest_ref = float(train_filled[nearest_col].quantile(0.50))
    margin_score = 1.0 / (1.0 + np.exp(-(valid_filled[margin_col] - margin_ref) / 20.0))
    entropy_score = 1.0 / (1.0 + np.exp((valid_filled[entropy_col] - entropy_ref) / 0.10))
    std_score = 1.0 / (1.0 + np.exp((valid_filled[std_col] - std_ref) / 5.0))
    boundary_score = 1.0 / (1.0 + np.exp((valid_filled[nearest_col] - nearest_ref) / 25.0))
    confidence = valid_filled[conf_col].clip(0.0, 1.0)
    raw = 0.30 * margin_score + 0.25 * entropy_score + 0.20 * std_score + 0.15 * confidence + 0.10 * boundary_score
    if schedule == "uncertainty_gate_open":
        raw = np.sqrt(np.clip(raw, 0.0, 1.0))
    return np.clip(max_weight * np.asarray(raw), 0.0, max_weight)

## 5. Pseudo-Top Gate

In [4]:
"Run fold-safe mean/residual/gate candidates."

mean_models = ["xgb_square_mean", "xgb_pseudohuber_mean", "huber_mean"]
residual_models = ["ridge_pseudo_state_residual", "huber_pseudo_state_residual"]
candidate_schedules = {
    "mean_only": {"type": "fixed", "weight": 0.0},
    "fixed_s010": {"type": "fixed", "weight": 0.10},
    "fixed_s020": {"type": "fixed", "weight": 0.20},
    "fixed_s030": {"type": "fixed", "weight": 0.30},
    "uncertainty_gate_conservative": {"type": "gate", "max_weight": 0.35},
    "uncertainty_gate_open": {"type": "gate", "max_weight": 0.50},
}


def evaluate_gated_regime(fold_col):
    residual_pseudo_cols, uncertainty_cols = pseudo_cols_for_fold(fold_col)
    residual_feature_cols = list(dict.fromkeys([*residual_pseudo_cols, *uncertainty_cols, *STATE_GATE_COLS]))
    gate_feature_cols = list(dict.fromkeys([f"pseudo_nearest_abs_offset__{fold_col}", f"pseudo_top_margin__{fold_col}", f"pseudo_top_confidence__{fold_col}", f"pseudo_mean_std__{fold_col}", f"pseudo_top_entropy__{fold_col}", *STATE_GATE_COLS]))
    X_base = data[BASE_FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    X_residual = data[residual_feature_cols].replace([np.inf, -np.inf], np.nan)
    X_gate = data[gate_feature_cols].replace([np.inf, -np.inf], np.nan)
    y_delta = data["target_delta"].astype(float)
    y_tvt = data["TVT"].astype(float)
    baseline = data["last_known_tvt_feature"].astype(float)
    fold_rows = []
    pred_parts = []
    gate_rows = []
    for fold in sorted(data[fold_col].dropna().unique()):
        valid_mask = data[fold_col].eq(fold)
        train_mask = ~valid_mask
        actual = y_tvt.loc[valid_mask].to_numpy()
        base_tvt = baseline.loc[valid_mask].to_numpy()
        for mean_name in mean_models:
            mean_train_delta, mean_train_model = fit_mean_predict(mean_name, X_base.loc[train_mask], y_delta.loc[train_mask], X_base.loc[train_mask])
            mean_valid_delta = mean_train_model.predict(X_base.loc[valid_mask]) if mean_name != "huber_mean" else mean_train_model.predict(X_base.loc[valid_mask])
            mean_valid_tvt = base_tvt + mean_valid_delta
            train_residual = y_delta.loc[train_mask].to_numpy() - mean_train_delta
            for residual_name in residual_models:
                residual_model = make_linear_residual(residual_name)
                residual_model.fit(X_residual.loc[train_mask], train_residual)
                valid_residual_raw = residual_model.predict(X_residual.loc[valid_mask])
                train_residual_raw = residual_model.predict(X_residual.loc[train_mask])
                for schedule_name, schedule in candidate_schedules.items():
                    if schedule["type"] == "fixed":
                        weights = np.full(valid_mask.sum(), schedule["weight"], dtype=float)
                    else:
                        weights = make_gate_weights(X_gate.loc[train_mask], X_gate.loc[valid_mask], schedule_name, schedule["max_weight"])
                    pred_tvt = mean_valid_tvt + weights * valid_residual_raw
                    label = f"{mean_name}__{residual_name}__{schedule_name}"
                    metrics = metric_row(actual, pred_tvt, base_tvt)
                    fold_rows.append({
                        "run_id": RUN_ID,
                        "fold_column": fold_col,
                        "fold": int(fold),
                        "label": label,
                        "mean_model": mean_name,
                        "residual_model": residual_name,
                        "schedule": schedule_name,
                        "n_train_rows": int(train_mask.sum()),
                        "n_valid_rows": int(valid_mask.sum()),
                        "n_base_features": int(len(BASE_FEATURE_COLS)),
                        "n_residual_features": int(len(residual_feature_cols)),
                        **metrics,
                    })
                    tmp = data.loc[valid_mask, ["wellname", PRIMARY_FOLD, STRESS_FOLD, "eval_progress", "selected_analog_distance", "combined_stress_score"]].copy()
                    tmp["fold_column"] = fold_col
                    tmp["fold"] = int(fold)
                    tmp["label"] = label
                    tmp["mean_model"] = mean_name
                    tmp["residual_model"] = residual_name
                    tmp["schedule"] = schedule_name
                    tmp["actual_tvt"] = actual
                    tmp["pred_tvt"] = pred_tvt
                    tmp["baseline_tvt"] = base_tvt
                    tmp["mean_pred_tvt"] = mean_valid_tvt
                    tmp["residual_raw"] = valid_residual_raw
                    tmp["gate_weight"] = weights
                    tmp["residual"] = actual - pred_tvt
                    pred_parts.append(tmp)
                    gate_rows.append({
                        "fold_column": fold_col,
                        "fold": int(fold),
                        "label": label,
                        "mean_model": mean_name,
                        "residual_model": residual_name,
                        "schedule": schedule_name,
                        "mean_gate_weight": float(np.mean(weights)),
                        "max_gate_weight": float(np.max(weights)),
                        "mean_abs_raw_correction": float(np.mean(np.abs(valid_residual_raw))),
                        "mean_abs_applied_correction": float(np.mean(np.abs(weights * valid_residual_raw))),
                        "train_residual_rmse_proxy": rmse(train_residual, train_residual_raw),
                    })
    return pd.DataFrame(fold_rows), pd.concat(pred_parts, ignore_index=True), pd.DataFrame(gate_rows)


start = time.time()
strict_fold_metrics, strict_predictions, strict_gate_diag = evaluate_gated_regime(PRIMARY_FOLD)
stress_fold_metrics, stress_predictions, stress_gate_diag = evaluate_gated_regime(STRESS_FOLD)
print(f"elapsed_s={time.time() - start:.1f}")
strict_fold_metrics.head()

C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


elapsed_s=292.2


,run_id,fold_column,fold,label,mean_model,residual_model,schedule,n_train_rows,n_valid_rows,n_base_features,n_residual_features,rmse,mae,r2,last_known_hold_rmse
0,2026-06-30__pseudo_top_gated_regime,strict_hybrid_fold,1,xgb_square_mean__ridge_pseudo_state_residual__...,xgb_square_mean,ridge_pseudo_state_residual,mean_only,61440,31320,41,42,13.941122,9.662122,0.999441,16.418628
1,2026-06-30__pseudo_top_gated_regime,strict_hybrid_fold,1,xgb_square_mean__ridge_pseudo_state_residual__...,xgb_square_mean,ridge_pseudo_state_residual,fixed_s010,61440,31320,41,42,13.941279,9.662738,0.999441,16.418628
2,2026-06-30__pseudo_top_gated_regime,strict_hybrid_fold,1,xgb_square_mean__ridge_pseudo_state_residual__...,xgb_square_mean,ridge_pseudo_state_residual,fixed_s020,61440,31320,41,42,13.941508,9.663450,0.999441,16.418628
3,2026-06-30__pseudo_top_gated_regime,strict_hybrid_fold,1,xgb_square_mean__ridge_pseudo_state_residual__...,xgb_square_mean,ridge_pseudo_state_residual,fixed_s030,61440,31320,41,42,13.941810,9.664257,0.999441,16.418628
4,2026-06-30__pseudo_top_gated_regime,strict_hybrid_fold,1,xgb_square_mean__ridge_pseudo_state_residual__...,xgb_square_mean,ridge_pseudo_state_residual,uncertainty_gate_conservative,61440,31320,41,42,13.941685,9.663869,0.999441,16.418628


## 6. Strict-CV Results

In [5]:
"Summarize strict and analog-distance results."

strict_summary = (
    strict_fold_metrics.groupby(["label", "mean_model", "residual_model", "schedule"])
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        r2_mean=("r2", "mean"),
        last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        n_valid_rows=("n_valid_rows", "sum"),
        n_base_features=("n_base_features", "max"),
        n_residual_features=("n_residual_features", "max"),
    )
    .reset_index()
)
stress_summary = (
    stress_fold_metrics.groupby(["label", "mean_model", "residual_model", "schedule"])
    .agg(
        stress_rmse_mean=("rmse", "mean"),
        stress_rmse_std=("rmse", "std"),
        stress_mae_mean=("mae", "mean"),
        stress_last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        stress_n_valid_rows=("n_valid_rows", "sum"),
    )
    .reset_index()
)
summary = strict_summary.merge(stress_summary, on=["label", "mean_model", "residual_model", "schedule"], how="left")
summary["rmse_delta_vs_hold"] = summary["rmse_mean"] - summary["last_known_hold_rmse_mean"]
summary["rmse_delta_vs_reference_safe_typewell"] = summary["rmse_mean"] - CONFIG["safe_typewell_reference_rmse"]
summary["rmse_delta_vs_public_best"] = summary["rmse_mean"] - CONFIG["active_public_best_rmse"]
summary["stress_minus_strict_rmse"] = summary["stress_rmse_mean"] - summary["rmse_mean"]
summary = summary.sort_values("rmse_mean").reset_index(drop=True)
best_row = summary.iloc[0].copy()
best_label = str(best_row["label"])

print(summary.head(18).to_string(index=False))

                                                                           label           mean_model              residual_model                      schedule  rmse_mean  rmse_std  mae_mean  r2_mean  last_known_hold_rmse_mean  n_valid_rows  n_base_features  n_residual_features  stress_rmse_mean  stress_rmse_std  stress_mae_mean  stress_last_known_hold_rmse_mean  stress_n_valid_rows  rmse_delta_vs_hold  rmse_delta_vs_reference_safe_typewell  rmse_delta_vs_public_best  stress_minus_strict_rmse
xgb_pseudohuber_mean__ridge_pseudo_state_residual__uncertainty_gate_conservative xgb_pseudohuber_mean ridge_pseudo_state_residual uncertainty_gate_conservative  13.469407  0.903485  9.515843 0.999519                  15.571277         92760               41                   42         13.766527         0.669037         9.842869                         15.581239                92760           -2.101870                               0.018807                  -1.002593                  0.297120
    

## 7. Stress Diagnostics

In [6]:
"Compute progress-zone, analog-distance, and well-level diagnostics."

predictions = pd.concat([strict_predictions, stress_predictions], ignore_index=True)
predictions["progress_zone"] = pd.cut(predictions["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])
predictions["analog_distance_bin"] = pd.qcut(predictions["selected_analog_distance"], q=5, labels=["closest", "near", "middle", "far", "farthest"], duplicates="drop")

zone_summary = (
    predictions.groupby(["fold_column", "label", "progress_zone"], observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "mean_rmse": rmse(group["actual_tvt"], group["mean_pred_tvt"]),
        "baseline_rmse": rmse(group["actual_tvt"], group["baseline_tvt"]),
        "mean_gate_weight": float(group["gate_weight"].mean()),
        "mean_abs_applied_correction": float(np.abs(group["gate_weight"] * group["residual_raw"]).mean()),
    }), include_groups=False)
    .reset_index()
)
zone_summary["rmse_delta_vs_mean"] = zone_summary["rmse"] - zone_summary["mean_rmse"]

analog_bin_summary = (
    predictions.groupby(["fold_column", "label", "analog_distance_bin"], observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "n_wells": group["wellname"].nunique(),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "mean_rmse": rmse(group["actual_tvt"], group["mean_pred_tvt"]),
        "baseline_rmse": rmse(group["actual_tvt"], group["baseline_tvt"]),
        "mean_gate_weight": float(group["gate_weight"].mean()),
    }), include_groups=False)
    .reset_index()
)
analog_bin_summary["rmse_delta_vs_mean"] = analog_bin_summary["rmse"] - analog_bin_summary["mean_rmse"]

well_diagnostics = (
    predictions.loc[(predictions["fold_column"].eq(PRIMARY_FOLD)) & (predictions["label"].eq(best_label))]
    .groupby("wellname")
    .agg(
        n_rows=("actual_tvt", "size"),
        strict_fold=(PRIMARY_FOLD, "first"),
        analog_fold=(STRESS_FOLD, "first"),
        mean_selected_analog_distance=("selected_analog_distance", "mean"),
        mean_gate_weight=("gate_weight", "mean"),
        mean_residual=("residual", "mean"),
        mean_abs_residual=("residual", lambda series: float(np.abs(series).mean())),
        rmse=("residual", lambda series: float(np.sqrt(np.mean(np.square(series))))),
    )
    .reset_index()
    .sort_values("rmse", ascending=False)
)

gate_diagnostics = pd.concat([strict_gate_diag, stress_gate_diag], ignore_index=True)
gate_summary = (
    gate_diagnostics.groupby(["fold_column", "label", "mean_model", "residual_model", "schedule"])
    .agg(
        mean_gate_weight=("mean_gate_weight", "mean"),
        max_gate_weight=("max_gate_weight", "max"),
        mean_abs_raw_correction=("mean_abs_raw_correction", "mean"),
        mean_abs_applied_correction=("mean_abs_applied_correction", "mean"),
        train_residual_rmse_proxy=("train_residual_rmse_proxy", "mean"),
    )
    .reset_index()
)

display(summary.head(12))

,label,mean_model,residual_model,schedule,rmse_mean,rmse_std,mae_mean,r2_mean,last_known_hold_rmse_mean,n_valid_rows,n_base_features,n_residual_features,stress_rmse_mean,stress_rmse_std,stress_mae_mean,stress_last_known_hold_rmse_mean,stress_n_valid_rows,rmse_delta_vs_hold,rmse_delta_vs_reference_safe_typewell,rmse_delta_vs_public_best,stress_minus_strict_rmse
0,xgb_pseudohuber_mean__ridge_pseudo_state_resid...,xgb_pseudohuber_mean,ridge_pseudo_state_residual,uncertainty_gate_conservative,13.469407,0.903485,9.515843,0.999519,15.571277,92760,41,42,13.766527,0.669037,9.842869,15.581239,92760,-2.101870,0.018807,-1.002593,0.297120
1,xgb_pseudohuber_mean__ridge_pseudo_state_resid...,xgb_pseudohuber_mean,ridge_pseudo_state_residual,uncertainty_gate_open,13.470234,0.898564,9.527619,0.999519,15.571277,92760,41,42,13.771735,0.673815,9.863544,15.581239,92760,-2.101042,0.019634,-1.001766,0.301501
2,xgb_pseudohuber_mean__ridge_pseudo_state_resid...,xgb_pseudohuber_mean,ridge_pseudo_state_residual,fixed_s020,13.470425,0.900776,9.516957,0.999519,15.571277,92760,41,42,13.771209,0.671628,9.846126,15.581239,92760,-2.100852,0.019825,-1.001575,0.300785
3,xgb_pseudohuber_mean__ridge_pseudo_state_resid...,xgb_pseudohuber_mean,ridge_pseudo_state_residual,fixed_s030,13.470435,0.898073,9.523110,0.999519,15.571277,92760,41,42,13.773497,0.674037,9.857172,15.581239,92760,-2.100842,0.019835,-1.001565,0.303062
4,xgb_pseudohuber_mean__huber_pseudo_state_resid...,xgb_pseudohuber_mean,huber_pseudo_state_residual,uncertainty_gate_open,13.471379,0.899556,9.512608,0.999519,15.571277,92760,41,42,13.798508,0.679301,9.863373,15.581239,92760,-2.099898,0.020779,-1.000621,0.327129
5,xgb_pseudohuber_mean__huber_pseudo_state_resid...,xgb_pseudohuber_mean,huber_pseudo_state_residual,fixed_s030,13.471873,0.900936,9.511548,0.999519,15.571277,92760,41,42,13.788082,0.673541,9.855213,15.581239,92760,-2.099404,0.021273,-1.000127,0.316209
6,xgb_pseudohuber_mean__ridge_pseudo_state_resid...,xgb_pseudohuber_mean,ridge_pseudo_state_residual,fixed_s010,13.472169,0.903208,9.512862,0.999518,15.571277,92760,41,42,13.771061,0.669595,9.837837,15.581239,92760,-2.099108,0.021569,-0.999831,0.298891
7,xgb_pseudohuber_mean__huber_pseudo_state_resid...,xgb_pseudohuber_mean,huber_pseudo_state_residual,uncertainty_gate_conservative,13.472640,0.902150,9.511152,0.999518,15.571277,92760,41,42,13.781325,0.670169,9.846474,15.581239,92760,-2.098637,0.022040,-0.999360,0.308686
8,xgb_pseudohuber_mean__huber_pseudo_state_resid...,xgb_pseudohuber_mean,huber_pseudo_state_residual,fixed_s020,13.472771,0.902421,9.510946,0.999518,15.571277,92760,41,42,13.780245,0.669873,9.845845,15.581239,92760,-2.098506,0.022171,-0.999229,0.307474
9,xgb_square_mean__ridge_pseudo_state_residual__...,xgb_square_mean,ridge_pseudo_state_residual,uncertainty_gate_open,13.473451,0.406479,9.554977,0.999522,15.571277,92760,41,42,13.831320,0.538806,10.035167,15.581239,92760,-2.097826,0.022851,-0.998549,0.357868


## 8. Artifact Export

In [7]:
"Export result artifacts, model comparison, and report."

best_strict_rmse = float(best_row["rmse_mean"])
best_stress_rmse = float(best_row["stress_rmse_mean"])
best_mean_only = summary.loc[summary["schedule"].eq("mean_only")].sort_values("rmse_mean").iloc[0]
best_gated = summary.loc[~summary["schedule"].eq("mean_only")].sort_values("rmse_mean").iloc[0]

if best_strict_rmse < CONFIG["safe_typewell_reference_rmse"] - 0.05 and best_stress_rmse <= CONFIG["safe_typewell_reference_stress_rmse"]:
    recommendation = "package_pseudo_top_gated_regime"
elif best_strict_rmse < CONFIG["active_public_best_rmse"]:
    recommendation = "iterate_before_package"
else:
    recommendation = "diagnostic_only"

findings = pd.DataFrame([
    {
        "finding": "best_overall",
        "detail": best_label,
        "value": best_strict_rmse,
        "interpretation": f"Best strict RMSE with analog stress RMSE {best_stress_rmse:.4f}.",
    },
    {
        "finding": "best_mean_only",
        "detail": str(best_mean_only["label"]),
        "value": float(best_mean_only["rmse_mean"]),
        "interpretation": "Best no-correction reference inside the same notebook.",
    },
    {
        "finding": "best_gated_or_fixed_residual",
        "detail": str(best_gated["label"]),
        "value": float(best_gated["rmse_mean"]),
        "interpretation": "Best candidate using residual correction with fixed or uncertainty-gated shrinkage.",
    },
    {
        "finding": "decision",
        "detail": recommendation,
        "value": best_strict_rmse,
        "interpretation": "Promotion rule checks safe/typewell local reference, analog stress, and public-best context.",
    },
])

paths = {
    "summary": TABLE_DIR / "model__pseudo_top_gated_regime_summary__2026-06-30.csv",
    "fold_metrics": TABLE_DIR / "model__pseudo_top_gated_regime_fold_metrics__2026-06-30.csv",
    "gate_diagnostics": TABLE_DIR / "model__pseudo_top_gated_regime_gate_diagnostics__2026-06-30.csv",
    "zone_summary": TABLE_DIR / "model__pseudo_top_gated_regime_zone_summary__2026-06-30.csv",
    "analog_bin_summary": TABLE_DIR / "model__pseudo_top_gated_regime_analog_bin_summary__2026-06-30.csv",
    "well_diagnostics": TABLE_DIR / "model__pseudo_top_gated_regime_well_diagnostics__2026-06-30.csv",
    "findings": TABLE_DIR / "model__pseudo_top_gated_regime_findings__2026-06-30.csv",
}
exports = {
    "summary": summary,
    "fold_metrics": pd.concat([strict_fold_metrics, stress_fold_metrics], ignore_index=True),
    "gate_diagnostics": gate_summary,
    "zone_summary": zone_summary,
    "analog_bin_summary": analog_bin_summary,
    "well_diagnostics": well_diagnostics,
    "findings": findings,
}
for key, frame in exports.items():
    frame.to_csv(paths[key], index=False)
    shutil.copy2(paths[key], RUN_DIR / paths[key].name)

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
new_row = pd.DataFrame([{
    "run_id": RUN_ID,
    "model_family": "alignment",
    "model_name": "pseudo_top_gated_regime",
    "best_candidate": best_label,
    "local_cv_rmse": best_strict_rmse,
    "local_cv_mae": float(best_row["mae_mean"]),
    "public_lb_rmse": np.nan,
    "status": recommendation,
    "notes": f"stress_rmse={best_stress_rmse:.4f}; best_mean_only={float(best_mean_only['rmse_mean']):.4f}; best_gated={float(best_gated['rmse_mean']):.4f}",
    "updated_at": RUN_DATE,
}])
if not comparison.empty and "run_id" in comparison.columns:
    comparison = comparison.loc[comparison["run_id"] != RUN_ID].copy()
comparison = pd.concat([comparison, new_row], ignore_index=True, sort=False)
comparison.to_csv(comparison_path, index=False)
shutil.copy2(comparison_path, RUN_DIR / comparison_path.name)

report_path = REPORT_DIR / "model__pseudo_top_gated_regime__2026-06-30.md"
report = f"""# Pseudo-Top Gated Regime Model

## Result

Best overall candidate: `{best_label}` with strict-CV RMSE {best_strict_rmse:.4f}, MAE {float(best_row["mae_mean"]):.4f}, and analog-distance stress RMSE {best_stress_rmse:.4f}.

Best mean-only candidate: `{best_mean_only["label"]}` with RMSE {float(best_mean_only["rmse_mean"]):.4f}.

Best residual-corrected candidate: `{best_gated["label"]}` with RMSE {float(best_gated["rmse_mean"]):.4f}.

## Gate Behavior

The gated candidates use pseudo-top margin, confidence, entropy, ensemble spread, and state/typewell descriptors to shrink residual corrections. Conservative gates clip weights at 0.35; open gates clip at 0.50.

## Leakage Notes

Direct formation columns are used only as fold-local labels for pseudo-top proxy training. Every pseudo-top feature is predicted for held-out rows by models trained on other folds.

## Decision

Recommendation: `{recommendation}`.

## Next Move

If residual gating improves over mean-only, package only after checking public-transfer risk; otherwise move the pseudo-top uncertainty into a stronger mean model or a Bayesian prior rather than another post-hoc residual head.
"""
report_path.write_text(report, encoding="utf-8")
shutil.copy2(report_path, RUN_DIR / report_path.name)

print(findings.to_string(index=False))
print(report_path)

                     finding                                                                           detail     value                                                                               interpretation
                best_overall xgb_pseudohuber_mean__ridge_pseudo_state_residual__uncertainty_gate_conservative 13.469407                                            Best strict RMSE with analog stress RMSE 13.7665.
              best_mean_only                          xgb_square_mean__ridge_pseudo_state_residual__mean_only 13.474489                                       Best no-correction reference inside the same notebook.
best_gated_or_fixed_residual xgb_pseudohuber_mean__ridge_pseudo_state_residual__uncertainty_gate_conservative 13.469407          Best candidate using residual correction with fixed or uncertainty-gated shrinkage.
                    decision                                                           iterate_before_package 13.469407 Promotion rule checks safe/t

## 9. Findings

In [8]:
"Show headline tables."

display(summary.head(15))
display(gate_summary.loc[gate_summary["label"].eq(best_label)].head())
display(findings)

,label,mean_model,residual_model,schedule,rmse_mean,rmse_std,mae_mean,r2_mean,last_known_hold_rmse_mean,n_valid_rows,n_base_features,n_residual_features,stress_rmse_mean,stress_rmse_std,stress_mae_mean,stress_last_known_hold_rmse_mean,stress_n_valid_rows,rmse_delta_vs_hold,rmse_delta_vs_reference_safe_typewell,rmse_delta_vs_public_best,stress_minus_strict_rmse
0,xgb_pseudohuber_mean__ridge_pseudo_state_resid...,xgb_pseudohuber_mean,ridge_pseudo_state_residual,uncertainty_gate_conservative,13.469407,0.903485,9.515843,0.999519,15.571277,92760,41,42,13.766527,0.669037,9.842869,15.581239,92760,-2.101870,0.018807,-1.002593,0.297120
1,xgb_pseudohuber_mean__ridge_pseudo_state_resid...,xgb_pseudohuber_mean,ridge_pseudo_state_residual,uncertainty_gate_open,13.470234,0.898564,9.527619,0.999519,15.571277,92760,41,42,13.771735,0.673815,9.863544,15.581239,92760,-2.101042,0.019634,-1.001766,0.301501
2,xgb_pseudohuber_mean__ridge_pseudo_state_resid...,xgb_pseudohuber_mean,ridge_pseudo_state_residual,fixed_s020,13.470425,0.900776,9.516957,0.999519,15.571277,92760,41,42,13.771209,0.671628,9.846126,15.581239,92760,-2.100852,0.019825,-1.001575,0.300785
3,xgb_pseudohuber_mean__ridge_pseudo_state_resid...,xgb_pseudohuber_mean,ridge_pseudo_state_residual,fixed_s030,13.470435,0.898073,9.523110,0.999519,15.571277,92760,41,42,13.773497,0.674037,9.857172,15.581239,92760,-2.100842,0.019835,-1.001565,0.303062
4,xgb_pseudohuber_mean__huber_pseudo_state_resid...,xgb_pseudohuber_mean,huber_pseudo_state_residual,uncertainty_gate_open,13.471379,0.899556,9.512608,0.999519,15.571277,92760,41,42,13.798508,0.679301,9.863373,15.581239,92760,-2.099898,0.020779,-1.000621,0.327129
5,xgb_pseudohuber_mean__huber_pseudo_state_resid...,xgb_pseudohuber_mean,huber_pseudo_state_residual,fixed_s030,13.471873,0.900936,9.511548,0.999519,15.571277,92760,41,42,13.788082,0.673541,9.855213,15.581239,92760,-2.099404,0.021273,-1.000127,0.316209
6,xgb_pseudohuber_mean__ridge_pseudo_state_resid...,xgb_pseudohuber_mean,ridge_pseudo_state_residual,fixed_s010,13.472169,0.903208,9.512862,0.999518,15.571277,92760,41,42,13.771061,0.669595,9.837837,15.581239,92760,-2.099108,0.021569,-0.999831,0.298891
7,xgb_pseudohuber_mean__huber_pseudo_state_resid...,xgb_pseudohuber_mean,huber_pseudo_state_residual,uncertainty_gate_conservative,13.472640,0.902150,9.511152,0.999518,15.571277,92760,41,42,13.781325,0.670169,9.846474,15.581239,92760,-2.098637,0.022040,-0.999360,0.308686
8,xgb_pseudohuber_mean__huber_pseudo_state_resid...,xgb_pseudohuber_mean,huber_pseudo_state_residual,fixed_s020,13.472771,0.902421,9.510946,0.999518,15.571277,92760,41,42,13.780245,0.669873,9.845845,15.581239,92760,-2.098506,0.022171,-0.999229,0.307474
9,xgb_square_mean__ridge_pseudo_state_residual__...,xgb_square_mean,ridge_pseudo_state_residual,uncertainty_gate_open,13.473451,0.406479,9.554977,0.999522,15.571277,92760,41,42,13.831320,0.538806,10.035167,15.581239,92760,-2.097826,0.022851,-0.998549,0.357868


,fold_column,label,mean_model,residual_model,schedule,mean_gate_weight,max_gate_weight,mean_abs_raw_correction,mean_abs_applied_correction,train_residual_rmse_proxy
22,analog_distance_fold,xgb_pseudohuber_mean__ridge_pseudo_state_resid...,xgb_pseudohuber_mean,ridge_pseudo_state_residual,uncertainty_gate_conservative,0.174445,0.314885,1.263085,0.230460,9.745524
58,strict_hybrid_fold,xgb_pseudohuber_mean__ridge_pseudo_state_resid...,xgb_pseudohuber_mean,ridge_pseudo_state_residual,uncertainty_gate_conservative,0.174703,0.313463,1.202455,0.217054,9.844049


,finding,detail,value,interpretation
0,best_overall,xgb_pseudohuber_mean__ridge_pseudo_state_resid...,13.469407,Best strict RMSE with analog stress RMSE 13.7665.
1,best_mean_only,xgb_square_mean__ridge_pseudo_state_residual__...,13.474489,Best no-correction reference inside the same n...
2,best_gated_or_fixed_residual,xgb_pseudohuber_mean__ridge_pseudo_state_resid...,13.469407,Best candidate using residual correction with ...
3,decision,iterate_before_package,13.469407,Promotion rule checks safe/typewell local refe...
